# 🌍 Global Temperature Anomalies — Exploratory Data Analysis

Monthly global mean surface temperature anomalies from GISS (GISTEMP) and NOAA (GCAG).

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Dark theme
plt.style.use('dark_background')
sns.set_theme(style='darkgrid')

print('Libraries loaded ✅')

## 1. Load & Inspect Data

In [ ]:
df = pd.read_csv(os.path.join('..', 'data', 'monthly.csv'))
print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head(10)

In [ ]:
df.info()

In [ ]:
df.describe()

## 2. Data Cleaning & Feature Engineering

In [ ]:
# Check for missing values
print('Missing values:')
print(df.isnull().sum())
print(f'\nDuplicate rows: {df.duplicated().sum()}')

In [ ]:
# Clean & create features
df = df.dropna(subset=['Source', 'Year', 'Mean'])
df['Mean'] = pd.to_numeric(df['Mean'], errors='coerce')
df = df.dropna(subset=['Mean'])

df['Date'] = pd.to_datetime(df['Year'], format='%Y-%m', errors='coerce')
df = df.dropna(subset=['Date'])
df['YearNum'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['MonthName'] = df['Date'].dt.strftime('%b')
df['Decade'] = (df['YearNum'] // 10 * 10).astype(str) + 's'

def get_season(m):
    if m in [12, 1, 2]: return 'Winter'
    elif m in [3, 4, 5]: return 'Spring'
    elif m in [6, 7, 8]: return 'Summer'
    else: return 'Autumn'

df['Season'] = df['Month'].apply(get_season)
df['AnomalyType'] = df['Mean'].apply(lambda x: 'Warm' if x > 0 else ('Cool' if x < 0 else 'Neutral'))

df = df.drop_duplicates().sort_values(['Date', 'Source']).reset_index(drop=True)
print(f'Cleaned shape: {df.shape}')
df.head()

## 3. Univariate Analysis

In [ ]:
# Distribution of anomaly values
fig, ax = plt.subplots(figsize=(10, 5))
for source, color in zip(['GCAG', 'GISTEMP'], ['#3B82F6', '#EF4444']):
    sub = df[df['Source'] == source]['Mean']
    ax.hist(sub, bins=40, alpha=0.6, color=color, label=source, edgecolor='#1E1E1E')
ax.axvline(x=0, color='#FBBF24', linestyle='--', linewidth=1.5, label='Zero baseline')
ax.set_xlabel('Temperature Anomaly (°C)')
ax.set_ylabel('Frequency')
ax.set_title('Distribution of Monthly Temperature Anomalies')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Summary statistics by source
df.groupby('Source')['Mean'].describe()

## 4. Bivariate Analysis

In [ ]:
# Trend over time by source
fig, ax = plt.subplots(figsize=(14, 5))
for source, color in zip(['GCAG', 'GISTEMP'], ['#3B82F6', '#EF4444']):
    sub = df[df['Source'] == source].groupby('YearNum')['Mean'].mean().reset_index()
    ax.plot(sub['YearNum'], sub['Mean'], color=color, linewidth=1.5, label=source, alpha=0.85)
ax.axhline(y=0, color='#FBBF24', linestyle=':', linewidth=1)
ax.set_xlabel('Year')
ax.set_ylabel('Mean Anomaly (°C)')
ax.set_title('Annual Mean Temperature Anomaly Trend')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Box plot by season
fig, ax = plt.subplots(figsize=(8, 5))
season_order = ['Winter', 'Spring', 'Summer', 'Autumn']
sns.boxplot(data=df, x='Season', y='Mean', order=season_order,
            palette=['#60A5FA', '#34D399', '#FBBF24', '#F97316'], ax=ax)
ax.axhline(y=0, color='#FBBF24', linestyle=':', alpha=0.5)
ax.set_title('Temperature Anomaly Distribution by Season')
ax.set_ylabel('Anomaly (°C)')
plt.tight_layout()
plt.show()

## 5. Multivariate Analysis

In [ ]:
# Correlation matrix
num_df = df[['YearNum', 'Month', 'Mean']].dropna()
corr = num_df.corr()

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(corr, annot=True, fmt='.3f', cmap='coolwarm', center=0, ax=ax,
            linewidths=0.5, vmin=-1, vmax=1)
ax.set_title('Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Average anomaly by decade
decade_avg = df.groupby('Decade')['Mean'].mean().reset_index().sort_values('Decade')

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#EF4444' if v > 0 else '#3B82F6' for v in decade_avg['Mean']]
ax.bar(decade_avg['Decade'], decade_avg['Mean'], color=colors, alpha=0.85)
ax.axhline(y=0, color='#FBBF24', linestyle='--', linewidth=1)
ax.set_xlabel('Decade')
ax.set_ylabel('Mean Anomaly (°C)')
ax.set_title('Average Temperature Anomaly by Decade')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 6. Key Findings

1. **Clear warming trend**: Temperature anomalies shift from consistently negative (pre-1940) to consistently positive (post-1980).
2. **Accelerating warming**: The rate of temperature increase accelerates sharply from the 1970s onward.
3. **Source agreement**: GCAG and GISTEMP show nearly identical trends, validating data reliability.
4. **Seasonal patterns**: All seasons show similar warming trends; no single season drives the overall signal.
5. **Strong Year-Anomaly correlation**: Year and anomaly value are positively correlated, confirming systematic global warming.
6. **Recent records**: The most extreme positive anomalies all occur in recent decades (2010s–2020s).